[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/AD7031/blob/main/notebooks/week02_linear_program.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "여기에 ACCESSID 붙여넣기",
    "WLSSECRET":   "여기에 SECRET 붙여넣기",
    "LICENSEID":   123456,   # 숫자 그대로, 따옴표 없이
}
env = gp.Env(params=params)

# First model — a toy LP

$$
\begin{aligned}
\max_{x,\,y} \quad & 4x + 5y \\
\text{s.t.} \quad & x + 3y \le 10 \\
                  & 3x + y \le 10 \\
                  & x,\, y \ge 0
\end{aligned}
$$

**Math to code**

- $x,\,y \ge 0$ — free; `addVar` lower bound is already `0`
- $\max\; 4x + 5y$ — `setObjective(expr, GRB.MAXIMIZE)`, expression **and** sense
- each $\le$ row — one `addConstr`, same coefficients, same order

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
# Initialize the model gp.Model()


# decision variables, we have two continuous variables --- e.g., lp.addVar(vtype=GRB.CONTINUOUS), lp.addVar(vtype=GRB.BINARY)


# let's define objective function --- lp.setObjective()


# now constraints, one per row of the math --- lp.addConstr()


# model is written down -- let's read it back before solving --- lp.update(), lp.write("warmup.lp"), print(open("warmup.lp").read())



In [ ]:
# Now that we are done with writing the optimization model on the computer, let's solve the problem --- lp.optimize()


# Let's print x, y and the objective value --- e.g., GREEN = "\033[92m"; RESET = "\033[0m"



## Reference: the completed model

In [ ]:
lp = gp.Model("warmup_lp")

# decision variables -- continuous, lower bound 0 by default
x = lp.addVar(vtype=GRB.CONTINUOUS, name="x")
y = lp.addVar(vtype=GRB.CONTINUOUS, name="y")

# objective -- max 4x + 5y
lp.setObjective(4 * x + 5 * y, GRB.MAXIMIZE)

# constraints -- one line per row of the math
lp.addConstr(x + 3 * y <= 10, name="c1")
lp.addConstr(3 * x + y <= 10, name="c2")

# read the model back before solving -- update() first, Gurobi builds it lazily
lp.update()
lp.write("warmup.lp")
print(open("warmup.lp").read())


In [ ]:
# Now that we are done with writing the optimization model on the computer, let's solve the problem
lp.optimize()

print(f"\nx = {x.X:.2f}   y = {y.X:.2f}")
print(f"objective = {lp.ObjVal:.2f}")


## Draw it

- **shaded** — feasible set: both constraints, plus $x,\,y \ge 0$
- **dashed** — objective $4x + 5y = 22.5$, pushed up-right until it leaves
- **star** — optimum $(2.5,\; 2.5)$

**The optimum sits on a corner.** True of every LP, at any number of variables.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

grid = np.linspace(0, 4, 400)
XX, YY = np.meshgrid(grid, grid)
feasible = ((XX + 3 * YY <= 10) & (3 * XX + YY <= 10)).astype(float)

plt.figure(figsize=(5.5, 5.5))
plt.contourf(XX, YY, feasible, levels=[0.5, 1.5], colors=["#AEC7E8"], alpha=0.7)
plt.plot(grid, (10 - grid) / 3, color="#4C72B0", label=r"$x + 3y \leq 10$")
plt.plot(grid, 10 - 3 * grid, color="#DD8452", label=r"$3x + y \leq 10$")

# the objective line through the optimum: 4x + 5y = ObjVal
plt.plot(grid, (lp.ObjVal - 4 * grid) / 5, "k--",
         label=rf"$4x + 5y = {lp.ObjVal:.1f}$")
plt.plot(x.X, y.X, "*", color="#C44E52", markersize=18, zorder=5)
plt.annotate(f"({x.X:.1f}, {y.X:.1f})", (x.X, y.X),
             textcoords="offset points", xytext=(10, 8))

plt.xlim(0, 4)
plt.ylim(0, 4)
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.title(f"Feasible region — optimum {lp.ObjVal:.2f} at a corner")
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()